# Notebook 9.1  A dialect identifier on self-supervised embeddings, with confusion analysis and a keyword-search demo

**Goal.** Train a simple Arabic dialect classifier on pretrained speech embeddings, read its confusion matrix, compute accuracy and macro-averaged F1, and search a small set of transcribed clips by keyword.

**How to use real data.** The cells run end to end on a small *synthetic* dataset so the notebook works with no downloads. Each step says how to swap in real audio: multidialect speech from [ADI17](https://arabicspeech.org/resources/adi17) or [Casablanca](https://arxiv.org/abs/2410.04527), and Saudi material from [SADA](https://doi.org/10.1109/ICASSP48485.2024.10446243), subject to their current licenses.

This notebook accompanies Chapter 9. It is deliberately small and CPU-only; the point is the *method* (embed, pool, classify, analyze), not state-of-the-art accuracy.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
rng = np.random.default_rng(0)
DIALECTS = ['MSA','Gulf','Egyptian','Levantine','Maghrebi']
print('dialects:', DIALECTS)

## 2. Embeddings: real encoder or synthetic fallback

In practice you extract one fixed-length vector per utterance from a self-supervised encoder (for example XLS-R) and pool over time. To keep this notebook self-contained, `embed_utterance` returns a synthetic embedding whose cluster depends on the dialect, with neighboring dialects placed closer together so the confusion structure of Chapter 9 appears.

**To use a real encoder**, replace the body with something like:
```python
import torch, torchaudio
bundle = torchaudio.pipelines.WAV2VEC2_XLSR53
model = bundle.get_model().eval()
wav, sr = torchaudio.load(path)
with torch.no_grad():
    feats, _ = model.extract_features(wav)
return feats[-1].mean(dim=1).squeeze().numpy()   # mean-pool the last layer
```

In [ ]:
DIM = 32
# place dialect centroids on a ring so neighbors are close (continuum)
angles = {d: 2*np.pi*i/len(DIALECTS) for i,d in enumerate(DIALECTS)}
centroids = {}
for d,a in angles.items():
    v = np.zeros(DIM)
    v[0], v[1] = np.cos(a), np.sin(a)        # 2-D continuum structure
    v += 0.15*rng.standard_normal(DIM)        # a little per-dialect character
    centroids[d] = v

def embed_utterance(dialect, spread=0.6):
    """Synthetic stand-in for a pooled self-supervised embedding."""
    return centroids[dialect] + spread*rng.standard_normal(DIM)

def make_dataset(n_per_class=120):
    X, y = [], []
    for d in DIALECTS:
        for _ in range(n_per_class):
            X.append(embed_utterance(d)); y.append(d)
    return np.array(X), np.array(y)

X, y = make_dataset()
print('dataset:', X.shape, '| labels:', Counter(y))

## 3. Train a lightweight classifier

A logistic-regression head on top of frozen embeddings is the standard cheap probe. We split speaker-disjoint in spirit by a simple random split here; with real data, split by speaker to avoid leakage (Chapter 7).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)
print('trained on', len(ytr), 'examples; tested on', len(yte))

## 4. Confusion matrix and the macro-F1 algorithm

We build the confusion matrix (rows = true, columns = predicted), then compute accuracy and macro-F1 with the exact five-step logic from Section 9.9 (TP on the diagonal, FP down the column, FN across the row, per-class precision/recall, class F1, then average). If both precision and recall are zero for a class, its F1 is defined as zero.

In [ ]:
def confusion_matrix(true, predicted, labels):
    idx = {l:i for i,l in enumerate(labels)}
    M = np.zeros((len(labels), len(labels)), dtype=int)
    for t,p in zip(true, predicted):
        M[idx[t], idx[p]] += 1
    return M

def accuracy_and_macro_f1(M):
    n = M.shape[0]
    acc = np.trace(M) / M.sum()
    f1s = []
    for c in range(n):
        tp = M[c, c]
        fp = M[:, c].sum() - tp      # rest of the column
        fn = M[c, :].sum() - tp      # rest of the row
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0   # zero-division rule
        f1s.append(f1)
    return acc, float(np.mean(f1s)), f1s

M = confusion_matrix(yte, pred, DIALECTS)
acc, macro_f1, per_class = accuracy_and_macro_f1(M)
print('confusion matrix (rows=true, cols=pred):')
print('        ' + '  '.join(f'{d[:4]:>5}' for d in DIALECTS))
for i,d in enumerate(DIALECTS):
    print(f'{d[:7]:>7} ' + '  '.join(f'{M[i,j]:>5}' for j in range(len(DIALECTS))))
print(f'\naccuracy = {acc:.3f}   macro-F1 = {macro_f1:.3f}')
for d,f in zip(DIALECTS, per_class): print(f'  F1[{d}] = {f:.3f}')

In [ ]:
# plot the confusion matrix
fig, ax = plt.subplots(figsize=(5.2,4.6))
im = ax.imshow(M, cmap='Blues')
ax.set_xticks(range(len(DIALECTS))); ax.set_xticklabels(DIALECTS, rotation=45, ha='right')
ax.set_yticks(range(len(DIALECTS))); ax.set_yticklabels(DIALECTS)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(DIALECTS)):
    for j in range(len(DIALECTS)):
        ax.text(j, i, M[i,j], ha='center', va='center', color='black', fontsize=9)
ax.set_title('Dialect confusion (neighbors confuse most)')
fig.tight_layout(); fig.savefig('dialect_confusion.png', dpi=120)
print('saved dialect_confusion.png')

### Why macro-F1, not just accuracy

Make one dialect rare and the gap appears: accuracy stays high while macro-F1 drops, because the rare class is weighted equally.

In [ ]:
# imbalanced test: shrink Maghrebi to a handful of examples
mask = ~((yte=='Maghrebi') & (np.arange(len(yte)) % 8 != 0))
M2 = confusion_matrix(yte[mask], pred[mask], DIALECTS)
a2, f2, _ = accuracy_and_macro_f1(M2)
print(f'imbalanced: accuracy = {a2:.3f}  macro-F1 = {f2:.3f}')
print('macro-F1 falls below accuracy when a rare dialect is handled worse than common ones')

## 5. A keyword-search demo over transcribed clips

Spoken document retrieval in miniature: index a few (transcript, dialect) records, then rank them against a keyword query. Arabic morphology means one query root appears in many forms, so we expand the query with simple clitic/prefix variants before matching (Section 9.6 to 9.7).

In [ ]:
clips = [
    {'id':'c1','dialect':'Gulf',     'text':'ابغى احجز موعد في العياده'},
    {'id':'c2','dialect':'Egyptian', 'text':'عايز احجز معاد في العياده'},
    {'id':'c3','dialect':'MSA',      'text':'اريد حجز موعد في المستشفى'},
    {'id':'c4','dialect':'Levantine','text':'بدي احكي مع الدكتور بكرا'},
    {'id':'c5','dialect':'Maghrebi', 'text':'بغيت نحجز رندي فو مع الطبيب'},
]

def variants(term):
    """Expand an Arabic query term with common attached prefixes (clitics)."""
    prefixes = ['', 'ال', 'و', 'ف', 'ب', 'لل']
    return {p+term for p in prefixes} | {term}

def search(query, clips):
    q = variants(query)
    scored = []
    for c in clips:
        toks = c['text'].split()
        hits = sum(any(tok.startswith(v) or v in tok for v in q) for tok in toks)
        if hits: scored.append((hits, c))
    return sorted(scored, key=lambda x: -x[0])

for q in ['موعد','احجز','العياده']:
    print(f'query = {q!r}')
    for hits, c in search(q, clips):
        print(f'   {c["id"]} [{c["dialect"]}] hits={hits}  {c["text"]}')
    print()

## 6. Where to go next

- Replace the synthetic embeddings with real XLS-R or HuBERT features over ADI17, Casablanca, or SADA, and split by speaker.
- Report accuracy and macro-F1 per dialect, and inspect the confusion matrix for neighbor-to-neighbor errors.
- Swap the keyword matcher for a subword or phonetic index so out-of-vocabulary names and dialectal spellings still match (Section 9.7).
- For dialect-aware ASR, use the classifier as a routing front end with a confidence threshold and an MSA or multidialect fallback (Section 9.9).